In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gdown
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l10/cars_new.csv'
DATA_FILE = 'cars_new.csv'


def load_cars(url=DATA_URL, file_name=DATA_FILE):
    gdown.download(url, file_name, quiet=True)
    data = pd.read_csv(file_name)
    data = data.dropna().copy()
    return data


def convert_price(value):
    number = ''.join(symbol for symbol in str(value) if symbol.isdigit())
    return int(number)


cars = load_cars()
cars['price'] = cars['price'].apply(convert_price)

mean_price = cars['price'].mean()
print(f'Средняя цена автомобиля в базе: {round(mean_price, 2)} руб.')

In [ ]:
CAT_FEATURES = ['mark', 'model', 'fuel', 'body']
NUM_FEATURES = ['year', 'mileage']
TARGET = 'price'


def build_dataset(table):
    encoded = pd.get_dummies(table[CAT_FEATURES])
    feature_block = np.hstack([encoded.values, table[NUM_FEATURES].values])

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    x_values = x_scaler.fit_transform(feature_block).astype('float32')
    y_values = y_scaler.fit_transform(table[[TARGET]].values).astype('float32')

    return x_values, y_values, x_scaler, y_scaler


x_data, y_data, scaler_x, scaler_y = build_dataset(cars)

In [ ]:
def split_samples(features, target):
    x_train_part, x_rest, y_train_part, y_rest = train_test_split(
        features,
        target,
        train_size=0.8
    )

    x_val_part, x_test_part, y_val_part, y_test_part = train_test_split(
        x_rest,
        y_rest,
        test_size=0.5
    )

    return x_train_part, x_val_part, x_test_part, y_train_part, y_val_part, y_test_part


x_train, x_val, x_test, y_train, y_val, y_test = split_samples(x_data, y_data)

print('Размер обучающей выборки:', x_train.shape)
print('Размер проверочной выборки:', x_val.shape)
print('Размер тестовой выборки:', x_test.shape)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def make_loader(x_values, y_values, batch_size=32, shuffle=False):
    x_tensor = torch.tensor(x_values, dtype=torch.float32)
    y_tensor = torch.tensor(y_values, dtype=torch.float32)
    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(x_train, y_train, batch_size=32, shuffle=True)
val_loader = make_loader(x_val, y_val, batch_size=32)
test_loader = make_loader(x_test, y_test, batch_size=32)

print('Устройство:', device)

In [ ]:
class CarPriceNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


model = CarPriceNet(x_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
def calculate_loss(model, loader, loss_function):
    model.eval()
    total_loss = 0
    total_count = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            prediction = model(x_batch)
            loss = loss_function(prediction, y_batch)
            batch_size = x_batch.size(0)
            total_loss += loss.item() * batch_size
            total_count += batch_size

    return total_loss / total_count


def train_model(model, train_loader, val_loader, epochs=50):
    history = {'loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0
        samples_count = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            prediction = model(x_batch)
            loss = criterion(prediction, y_batch)
            loss.backward()
            optimizer.step()

            batch_size = x_batch.size(0)
            running_loss += loss.item() * batch_size
            samples_count += batch_size

        train_loss = running_loss / samples_count
        val_loss = calculate_loss(model, val_loader, criterion)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Эпоха {epoch:02d}/50 | loss: {train_loss:.5f} | val_loss: {val_loss:.5f}')

    return history


history = train_model(model, train_loader, val_loader, epochs=50)

In [ ]:
def show_loss_graph(history):
    plt.figure(figsize=(10, 5))
    plt.plot(history['loss'], label='Ошибка обучения')
    plt.plot(history['val_loss'], label='Ошибка проверки')
    plt.xlabel('Эпоха')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)
    plt.show()


show_loss_graph(history)

In [ ]:
def predict_scaled(model, features):
    model.eval()
    values = torch.tensor(features, dtype=torch.float32).to(device)
    with torch.no_grad():
        prediction = model(values).cpu().numpy()
    return prediction


def restore_target(values, scaler):
    return scaler.inverse_transform(values).reshape(-1)


def estimate_result(model, x_values, y_values, target_scaler, base_price):
    prediction_scaled = predict_scaled(model, x_values)
    predicted_price = restore_target(prediction_scaled, target_scaler)
    real_price = restore_target(y_values, target_scaler)

    mae = np.mean(np.abs(predicted_price - real_price))
    error_percent = mae / base_price * 100

    return predicted_price, real_price, mae, error_percent


pred_price, real_price, mae, error_percent = estimate_result(
    model,
    x_test,
    y_test,
    scaler_y,
    mean_price
)

print(f'Средняя ошибка (MAE): {round(mae, 2)} руб.')
print(f'Суммарный процент ошибки: {round(error_percent, 2)}%')

In [ ]:
def draw_price_scatter(real_values, predicted_values):
    limit = max(real_values.max(), predicted_values.max())

    plt.figure(figsize=(8, 8))
    plt.scatter(real_values, predicted_values, alpha=0.3)
    plt.plot([0, limit], [0, limit], 'r')
    plt.xlabel('Реальная цена')
    plt.ylabel('Предсказанная цена')
    plt.grid(True)
    plt.show()


draw_price_scatter(real_price, pred_price)